In [3]:
# Importing libraries
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from draw_widgets import DrawWidgets
from ipyfilechooser import FileChooser
import os

dw = DrawWidgets()

## Run the cell below in order to generate the buttons to load the matrices

In [ ]:
fileX = widgets.FileUpload(
    accept="",  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    description="X matrix",
)
display(fileX)
filey = widgets.FileUpload(
    accept="",  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    description="y vector",
)
display(filey)

## Run the cell below in order to adequate the matrices to the correct format used for calculations

In [ ]:
X = dw.mountMatrix(fileX).values
y = dw.mountyvector(filey)

## Run the cell Below to perform a cross-validation

In [ ]:
# Run Cross-validation
from modules.cross_validation_class import CrossValidation

cv = CrossValidation(X, y)
print(cv.Q2())

## Run the cell below to see a plot of R² and Q² values versus latent variables

In [ ]:
# Plot R² and Q² for different number of latent variables
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

output_notebook()

source = ColumnDataSource(
    data=dict(
        lv=range(1, 12),
        R2=cv.R2(),
        Q2=cv.Q2(),
    )
)

TOOLTIPS = [("LV", "@lv"), ("R2", "@R2"), ("Q2", "@Q2")]

p = figure(
    title="Cross-validation error - PLS",
    x_axis_label="Latent variables number",
    y_axis_label="Q² or R²",
    tooltips=TOOLTIPS,
)
# p.text(x+0.3,y+0.3,flav.index[i])
p.line("lv", "R2", legend="R²", source=source)
p.circle("lv", "R2", legend="R²", source=source)
p.line("lv", "Q2", color="red", legend="Q²", source=source)
p.circle("lv", "Q2", color="red", legend="Q²", source=source)
show(p)

## Choose the number of latent variables. The pre selected value is the optimal one

In [ ]:
nLV_widget = dw.drawIntSlider(
    value=np.argmax(cv.Q2()) + 1,
    min=1,
    max=min(X.shape),
    description="Number of latent variables:",
    width="300pt",
)
display(nLV_widget)

## Run the cell below to capture the number of latent variables

In [ ]:
nLV = nLV_widget.value

# Run the cell below to see a summary of the cross-validation results

In [ ]:
# See a summary of the cross-validation results
cv.returnParameters(nLV)

## Run the cell below to plot experimental X predicted values of y for calibration

In [ ]:
# Plot experimental X predicted values of y for calibration
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

output_notebook()

source = ColumnDataSource(
    data=dict(
        y=y,
        y_pred=cv.ycal[:, nLV - 1],
    )
)

TOOLTIPS = [("y", "@y"), ("y_pred", "@y_pred")]

p = figure(
    title="Calibration prediction",
    x_axis_label="Experimental pIC50",
    y_axis_label="Predicted pIC50",
    tooltips=TOOLTIPS,
)
# p.text(x+0.3,y+0.3,flav.index[i])
p.circle("y", "y_pred", source=source)
p.line(y[:, 0], y[:, 0], color="red")
show(p)

## Run the cell below to plot experimental X predicted values of y for cross-validation

In [ ]:
# Plot experimental X predicted values of y for cross-validation
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

output_notebook()

source = ColumnDataSource(
    data=dict(
        y=y,
        y_pred=cv.ycv[:, nLV - 1],
    )
)

TOOLTIPS = [("y", "@y"), ("y_pred", "@y_pred")]

p = figure(
    title="Cross-validation prediction",
    x_axis_label="Experimental pIC50",
    y_axis_label="Predicted pIC50",
    tooltips=TOOLTIPS,
)
# p.text(x+0.3,y+0.3,flav.index[i])
p.circle("y", "y_pred", source=source)
p.line(y[:, 0], y[:, 0], color="red")
show(p)

## Run the cell below to choose the directory you want to save the results and type the filename you want

In [ ]:
# Create and display a FileChooser widget
fc = FileChooser(os.getcwd())
display(fc)

## Run the cell below to save the file in the directory selected with the name you typed

In [ ]:
# Save cross validation results file
cv.saveParameters(fc.selected, nLV)

# y-Randomization

## Choose the number of randomiztions you want to perform

In [ ]:
nrand_widget = dw.drawIntSlider(
    value=50, min=1, max=200, description="Number of randomizations:", width="400pt"
)
display(nrand_widget)

## Run the cell below to run y-randomization

In [ ]:
n_randomizations = nrand_widget.value

# y-randomization
from qsarmodelingpy.yrandomization import YRandomization

yr = YRandomization(X, y, nLV, n_randomizations)

## Plot R² and Q² values obtained in y-randomization

In [ ]:
# Plot R² and Q² values obtained in y-randomization
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

output_notebook()

source = ColumnDataSource(
    data=dict(
        R2=yr.R2,
        Q2=yr.Q2,
    )
)

TOOLTIPS = [("R2", "@R2"), ("Q2", "@Q2")]

p = figure(
    title="y-randomization", x_axis_label="R²", y_axis_label="Q²", tooltips=TOOLTIPS
)
# p.text(x+0.3,y+0.3,flav.index[i])
p.circle("R2", "Q2", source=source)
show(p)

## Plot Correlation between y randomized values and real y against R² and Q²

In [ ]:
# Plot Correlation between y randomized values and real y against R² and Q² according to
# Eriksson, L., Jaworska, J., Worth, A. P., Cronin, M. T. D., McDowell, R. M., & Gramatica, P. (2003).
# Methods for reliability and uncertainty assessment and for applicability evaluations of classification- and
# regression-based QSARs. Environmental Health Perspectives, 111(10), 1361–1375. https://doi.org/10.1289/ehp.5758
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource
from bokeh.layouts import gridplot

output_notebook()

source1 = ColumnDataSource(
    data=dict(
        R2=yr.R2,
        R=yr.R,
    )
)

aR2, bR2 = yr.returnRegResultsR2()

xR2 = np.linspace(np.min(yr.R), np.max(yr.R))
yR2 = aR2[0] * xR2 + bR2

TOOLTIPS = [("R", "@R"), ("R2", "@R2")]

s1 = figure(
    title="y-randomization", x_axis_label="R", y_axis_label="R²", tooltips=TOOLTIPS
)
# p.text(x+0.3,y+0.3,flav.index[i])
s1.circle("R", "R2", source=source1)
s1.line(xR2, yR2, color="red", legend="{:.2f}x + {:.2f}".format(aR2[0][0], bR2[0]))
s1.legend.location = "top_left"

source2 = ColumnDataSource(
    data=dict(
        Q2=yr.Q2,
        R=yr.R,
    )
)

aQ2, bQ2 = yr.returnRegResultsQ2()

xQ2 = np.linspace(np.min(yr.R), np.max(yr.R))
yQ2 = aQ2[0] * xQ2 + bQ2

TOOLTIPS = [("R", "@R"), ("Q2", "@Q2")]

s2 = figure(
    title="y-randomization", x_axis_label="R", y_axis_label="Q²", tooltips=TOOLTIPS
)
# p.text(x+0.3,y+0.3,flav.index[i])
s2.circle("R", "Q2", source=source2)
s2.line(xQ2, yQ2, color="red", legend="{:.2f}x + {:.2f}".format(aQ2[0][0], bQ2[0]))
s2.legend.location = "top_left"

p = gridplot([[s1, s2]])

show(p)

## Run the cell below to choose the directory you want to save the y-randomization results and type the filename with .csv extension

In [ ]:
from ipyfilechooser import FileChooser
import os

# Create and display a FileChooser widget
fcyr = FileChooser(os.getcwd())
display(fcyr)

## Run the cell bellow to save the y-randomization results in the selected file

In [ ]:
# Saving y-randomization results

yrMatrix = [yr.R2, yr.Q2, yr.R]
dfyr = pd.DataFrame(columns=["R2", "Q2", "R(yrd,y)"], data=np.transpose(yrMatrix))
dfyr.to_csv(fcyr.selected, sep=",", index=False)

# Leave-N-Out

## Choose the number of the maximum N value. The values corresponding to 25% of samples is already selected

In [ ]:
nlno_widget = dw.drawIntSlider(
    value=int(0.25 * X.shape[0]),
    min=1,
    max=int(X.shape[0] / 2),
    description="Maximum N value:",
    width="250pt",
)
display(nlno_widget)

## Choose the number of repetitions for each N value

In [ ]:
nrep_widget = dw.drawIntSlider(
    value=5, min=1, max=10, description="Number of repetitions:", width="250pt"
)
display(nrep_widget)

## Run the cell below to execute leave-N-out test

In [ ]:
# Leave-N-out
from qsarmodelingpy.lno import LNO

# Here you can choose the number of repetitions
N = nlno_widget.value
n_repetitions = nrep_widget.value

lno = LNO(X, y, nLV, n=N, nrepet=n_repetitions)
dfLNO = pd.DataFrame(data=lno.Q2)

## Plot the graph with leave-N-out results

In [ ]:
# Plot the graph with leave-N-out results
from bokeh.models import ColumnDataSource, Whisker, Range1d
from bokeh.plotting import figure, show, output_notebook
from bokeh.sampledata.autompg import autompg as df

output_notebook()

p = figure(
    title="Leave-N-Out",
    x_axis_label="N",
    y_axis_label="Q²",
)

lno_mean = dfLNO.mean(1)
lno_std = dfLNO.std(1)

base = range(1, dfLNO.shape[0] + 1)
upper = lno_mean + lno_std
lower = lno_mean - lno_std

source_error = ColumnDataSource(data=dict(base=base, lower=lower, upper=upper))

p.add_layout(Whisker(source=source_error, base="base", upper="upper", lower="lower"))

p.circle(x=base, y=lno_mean, color="black")

p.x_range = Range1d(0, len(base) + 1)
p.y_range = Range1d(np.min(lower) - 0.1, np.max(lower) + 0.1)

show(p)

## Run the cell below to choose the directory you want to save the leave-N-out results and type the filename with .csv extension¶

In [ ]:
from ipyfilechooser import FileChooser
import os

# Create and display a FileChooser widget
fclno = FileChooser(os.getcwd())
display(fclno)

## Run the cell bellow to save the y-randomization results in the selected file

In [ ]:
# Saving Leave-N-out results
dfLNO.index = ["Leave-{}-out".format(i + 1) for i in range(len(dfLNO))]
dfLNO.columns = ["Repetition {}".format(i + 1) for i in range(dfLNO.shape[1])]
dfLNO.to_csv(fclno.selected)